# 📈 Step 6: Demand Forecasting

## Overview
Train time series models for demand forecasting. We use simpler ML models (more suitable for 180K observations) alongside LSTM.

**ML Models (Recommended for this dataset size):**
- Ridge/Lasso Regression
- Random Forest, Gradient Boosting, XGBoost
- Extra Trees

**Deep Learning (Optional):**
- LSTM (in separate module)

**Feature Engineering for Time Series:**
- Lag features (previous days' demand)
- Rolling statistics (mean, std, min, max)
- Temporal features (day of week, month, etc.)

---


In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.data.preprocess import load_and_preprocess
from src.models.forecaster import DemandForecaster
plt.style.use('seaborn-v0_8-whitegrid')
print("✅ Setup complete!")


In [ ]:
# Load data
df = load_and_preprocess()
print(f"Data: {df.shape}")


## 6.1 Train ML Forecasting Models


In [ ]:
# Initialize forecaster
forecaster = DemandForecaster(random_state=42)
forecaster.initialize_models()

# Prepare time series features
X, y, dates = forecaster.prepare_time_series_features(df)
X_train, X_test, y_train, y_test = forecaster.split_time_series(X, y)

# Train all models
forecaster.train_all_models(X_train, y_train, X_test, y_test)


## 6.2 Model Comparison & Overfitting Analysis


In [ ]:
# Model comparison
comparison_df = forecaster.get_comparison_dataframe()
print("MODEL COMPARISON:")
print(comparison_df.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
models = comparison_df['Model'].values
x = np.arange(len(models))
width = 0.35
ax.bar(x - width/2, comparison_df['Train R²'], width, label='Train R²', color='steelblue')
ax.bar(x + width/2, comparison_df['Test R²'], width, label='Test R²', color='coral')
ax.set_ylabel('R² Score')
ax.set_title('Train vs Test R² - Overfitting Check', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=45, ha='right')
ax.legend()
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print("\n💡 ML models are faster & often better than LSTM for this data size")


## 6.3 Feature Importance for Forecasting


In [ ]:
# Feature importance
importance_df = forecaster.get_feature_importance(X.columns.tolist(), top_n=15)
if importance_df is not None:
    fig, ax = plt.subplots(figsize=(10, 6))
    importance_df.sort_values('importance').plot(x='feature', y='importance', kind='barh', ax=ax, legend=False, color='steelblue')
    ax.set_xlabel('Importance')
    ax.set_title(f'Top Features - {forecaster.best_model_name}', fontweight='bold')
    plt.tight_layout()
    plt.show()

print("\n💡 Lag features (previous days) are most important for forecasting")


## 6.4 (Optional) LSTM Training

For deep learning approach, run the LSTM module separately:


In [ ]:
# Optional: Run LSTM (takes longer)
# from src.models.train_lstm import run_lstm_training_pipeline
# lstm_forecaster, lstm_results = run_lstm_training_pipeline()

print("💡 LSTM Note: For 180K observations, ML models are often more practical.")
print("   LSTM excels with smaller datasets where patterns are complex.")
print("   Use: python main.py --train-lstm to run LSTM separately")


In [ ]:
# Save models
forecaster.save_models()
print("\n✅ Forecasting complete!")
print("➡️ Next: Run 07_model_evaluation.ipynb")
